# 04 · Feature Engineering

**Phase goal:** turn raw columns into a compact, leakage-safe feature matrix every model family can consume.

## The core decision: target-encode `make`/`model`, don't one-hot them
With ~3,200 distinct models, one-hot encoding produces a ~3,200-column matrix — huge, slow, and unusable by `HistGradientBoosting` (no sparse input). **Target encoding** maps each category to its (cross-fitted) mean price → just **two** dense numeric columns, no leakage.

In [1]:
import sys, warnings
sys.path.insert(0, '../src')
warnings.filterwarnings('ignore')
import pandas as pd
from car_pricing import config, data, features
df = data.clean(data.load_raw())
onehot_cols = df['make'].nunique() + df['model'].nunique()
print(f'One-hot make+model would add ~{onehot_cols:,} columns')
print(f'Target encoding adds exactly {len(config.TARGET_ENCODE)} columns')
print(f'Total production features: {len(config.FEATURES)}')

One-hot make+model would add ~3,274 columns
Target encoding adds exactly 2 columns
Total production features: 16


In [2]:
# Build the preprocessor and transform a sample to see the compact output
X, y = features.split_xy(df)
pre = features.build_preprocessor()
Xt = pre.fit_transform(X, y)
print('encoded matrix shape:', Xt.shape)
pd.DataFrame(Xt[:3], columns=config.FEATURES).round(2)

encoded matrix shape: (19820, 16)


,make,model,km_driven,mileage,engine,max_power,age,Individual,Trustmark Dealer,Diesel,Electric,LPG,Petrol,Manual,Seats_5,Seats_Above_5
0,4.70,1.14,120000.0,19.7,796.0,46.3,11.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0
1,5.43,4.73,20000.0,18.9,1197.0,82.0,7.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0
2,5.44,3.50,60000.0,17.0,1197.0,80.0,13.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0


In [3]:
# Tercile band edges (in Lakhs) used to DERIVE the band from a price
edges = features.band_edges(y)
print('band edges:', edges)
print('example:', features.price_to_band([2.0, 5.0, 25.0], edges))

band edges: [0.3, 3.99, 6.75, 20.9]
example: ['Low' 'Medium' 'High']


### Why this is production-grade
- **Compact:** 16 features, not ~3,200 → the trained artifact is ~1 MB, not 100+ MB.
- **Leakage-safe:** `TargetEncoder` cross-fits internally, and in CV we fit it *inside each fold* (see nb 05 / `train.py`).
- **Universal:** dense output works for every model, including boosting.
- **One contract:** the encoder lives *inside* the sklearn Pipeline, so the exact same transform is applied at train and serve time.